# AAG agents on a real model

Runs the AAG language-model agents (A1 evidence ingestion, A3 re-verification planning, and a
model-alone change-impact baseline) on a sample of the 19,500 benchmark scenarios, then scores them.

**Before you start:** the Colab secret `ANTHROPIC_API_KEY` must exist with notebook access on (key icon on the left).

**To run:** Runtime > Run all. Allow Google Drive access when asked. A CPU runtime is enough.
It sends requests through the Message Batches API (half price) and stops at the budget below.
Results go to `MyDrive/aag-agent-eval/`. If Colab disconnects, run all again: it resumes where it stopped
and does not pay twice for finished batches.

In [ ]:
BUDGET_USD = 18.0          # hard ceiling for this notebook (pilot + main run)
MODEL = "claude-opus-5-5"  # pinned model id
!pip -q install anthropic==2.11.0

In [ ]:
import os, sys, json, time
from google.colab import drive, userdata
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/aag-agent-eval'
os.makedirs(OUT, exist_ok=True)
os.makedirs('/content/aag', exist_ok=True); os.chdir('/content/aag'); sys.path.insert(0, '/content/aag')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('Drive folder:', OUT)

In [ ]:
%%writefile /content/aag/bench.py
#!/usr/bin/env python3
"""Change-impact benchmark for the Agentic Assurance Graph (AAG).

Ground truth is semantic: after a change, every analysis is re-run on the changed system
and an evidence item counts as stale iff its output differs. Methods only see artifact
hashes (syntactic). Analyses are real but small:
  - multicore interference: per-access delay bound from round-robin arbitration on the
    shared island SRAM port (slot latency x number of contending cores);
  - response-time analysis per core: fixed-priority (rate-monotonic) RTA with
    interference-inclusive execution times C' = C + m*d (m = accesses per job);
  - MPU partition check over island partitions (overlap = violation);
  - HIL latency test of the brake chain (RTA with average-case interference, 10 us resolution);
  - TARA: risk per threat from attack feasibility, controls and CAN access surface;
  - unit tests (behaviour version) and static analysis (code version) per task;
  - end-to-end protection tests (E2E library and protected CAN messages).
No language model is involved. Usage: python3 bench.py [n_systems] [seed] > results.json
"""
import hashlib, json, math, random, sys, copy
from collections import defaultdict

ISLAND = ["I0", "I1"]
APP = ["A0", "A1", "A2", "A3"]
CORES = ISLAND + APP
# Kramer et al. (WATERS 2015) period shares for periodic runnables, angle-synchronous excluded
PERIODS_MS = [1, 2, 5, 10, 20, 50, 100, 200, 1000]
PERIOD_W = [3, 2, 2, 25, 25, 3, 20, 1, 4]
COST_H = {"interference.analytic": 2, "interference.measured": 8, "rta": 0.5, "mpu": 0.1,
          "hil": 16, "unit": 0.1, "static": 0.2, "e2e": 0.5, "tara": 24, "review": 8}

def hsh(v):
    return hashlib.sha256(repr(v).encode()).hexdigest()[:16]

def uunifast(n, u, rng):
    out, s = [], u
    for i in range(1, n):
        nxt = s * rng.random() ** (1.0 / (n - i))
        out.append(s - nxt); s = nxt
    out.append(s)
    return out

# ----------------------------------------------------------------------------- system
def gen_system(rng):
    tasks = {}
    for core in CORES:
        n = rng.randint(3, 6)
        u = rng.uniform(0.35, 0.55) if core in ISLAND else rng.uniform(0.4, 0.65)
        for i, ui in enumerate(uunifast(n, u, rng)):
            T = rng.choices(PERIODS_MS, PERIOD_W)[0] * 1000
            name = f"{core}t{i}"
            tasks[name] = dict(core=core, C=max(5.0, round(ui * T, 1)), T=T,
                               r=round(rng.uniform(0.3, 2.0), 2),
                               sram=(core in ISLAND) or rng.random() < 0.25,
                               crit=("D" if core == "I0" else "B") if core in ISLAND else "QM",
                               code=0, behav=0, can=False)
    # the brake controller: highest-rate task on I0
    tasks["BrakeCtl"] = dict(core="I0", C=round(rng.uniform(150, 400), 1), T=5000, r=1.0, sram=True,
                             crit="D", code=0, behav=0, can=True)
    tasks["WSpeed"] = dict(core="I1", C=round(rng.uniform(100, 300), 1), T=5000, r=0.8, sram=True,
                           crit="B", code=0, behav=0, can=True)
    parts, base = {}, 0
    for t in sorted(x for x in tasks if tasks[x]["core"] in ISLAND):
        size = rng.choice([4, 8, 16]) * 1024
        parts[t] = (base, size); base += size + rng.choice([0, 1024, 4096])
    can = {f"msg{i}": rng.random() < 0.6 for i in range(rng.randint(4, 8))}
    can["BrakeReq"] = True
    return dict(tasks=tasks, parts=parts, ic=dict(L=round(rng.uniform(0.02, 0.06), 3),
                app_port=rng.random() < 0.5), controls=["SecOC"], can=can, e2e_code=0, item=0,
                dma=False)

# ----------------------------------------------------------------------------- analyses
def interference(s):
    """Per-access delay bound on each core: round-robin arbitration, each other requester j
    delays an access by at most one slot L, weighted by its access intensity min(1, rho_j)
    (rho_j = SRAM accesses per microsecond issued by core j)."""
    rho = defaultdict(float)
    for t, d in s["tasks"].items():
        if d["sram"] and (d["core"] in ISLAND or s["ic"]["app_port"]):
            rho[d["core"]] += d["C"] / d["T"] * d["r"]
    if s.get("dma"):
        rho["DMA"] = 0.8   # an app-side DMA engine reaching the SRAM port (in no template)
    return {k: round(s["ic"]["L"] * sum(min(1.0, v) for j, v in rho.items() if j != k), 4)
            if k in rho else 0.0 for k in CORES}

def rta(s, core, d, scale=1.0):
    ts = sorted([t for t, x in s["tasks"].items() if x["core"] == core],
                key=lambda t: (s["tasks"][t]["T"], t))
    out, hp = [], []
    for t in ts:
        x = s["tasks"][t]
        c = x["C"] + x["r"] * x["C"] * d * scale
        R = c
        for _ in range(1000):
            Rn = c + sum(math.ceil(R / s["tasks"][j]["T"]) * cj for j, cj in hp)
            if Rn > x["T"]:
                R = None; break
            if abs(Rn - R) < 1e-9:
                break
            R = Rn
        out.append((t, "miss" if R is None else round(R, 1)))
        hp.append((t, c))
    return tuple(out)

def mpu(s):
    p = sorted(s["parts"].items())
    viol = []
    for i in range(len(p)):
        for j in range(i + 1, len(p)):
            (a, (ba, sa)), (b, (bb, sb)) = p[i], p[j]
            if ba < bb + sb and bb < ba + sa:
                viol.append((a, b))
    return (tuple(t for t, _ in p), tuple(viol))

def hil(s, d):
    r = dict(rta(s, "I0", d["I0"], scale=0.5)).get("BrakeCtl")
    return "miss" if r == "miss" else int(round((r + 300) / 10.0)) * 10

def tara(s):
    risks = []
    surface = sum(1 for x in s["tasks"].values() if x["core"] in ISLAND and x["can"])
    for m, prot in sorted(s["can"].items()):
        feas = 3 - ("IDS" in s["controls"]) - (1 if prot and "SecOC" in s["controls"] else 0)
        feas += 1 if surface > 2 else 0
        risks.append((m, max(feas, 0) * (3 if m == "BrakeReq" else 2)))
    risks.append(("OTA", 2 if "SecureBoot" in s["controls"] else 4))
    return tuple(risks)

def outputs(s, ev_ids):
    d = interference(s)
    out = {}
    for e in ev_ids:
        kind, _, arg = e.partition(":")
        if kind == "INT":
            out[e] = tuple(sorted(d.items()))
        elif kind == "RTA":
            out[e] = rta(s, arg, d[arg])
        elif kind == "MPU":
            out[e] = mpu(s)
        elif kind == "HIL":
            out[e] = hil(s, d)
        elif kind == "UT":
            out[e] = s["tasks"][arg]["behav"] if arg in s["tasks"] else None
        elif kind == "SA":
            out[e] = s["tasks"][arg]["code"] if arg in s["tasks"] else None
        elif kind == "E2E":
            out[e] = (s["e2e_code"], tuple(sorted(m for m, p in s["can"].items() if p)))
        elif kind == "TARA":
            out[e] = tara(s)
        elif kind == "HARA":
            out[e] = s["item"]
    return out

def critical(e, old, new):
    """A change in output that worsens a verdict."""
    kind = e.split(":")[0]
    if kind == "RTA":
        return sum(1 for _, r in new if r == "miss") > sum(1 for _, r in old if r == "miss")
    if kind == "MPU":
        return len(new[1]) > len(old[1])
    if kind == "HIL":
        return new == "miss" or (old != "miss" and new > old)
    if kind == "TARA":
        o, n = dict(old), dict(new)
        return any(k not in o or n[k] > o[k] for k in n)
    if kind in ("UT", "E2E"):
        return True
    return False

# ----------------------------------------------------------------------------- artifacts
def artifacts(s, file_level=False):
    a = {}
    T = s["tasks"]
    rows = {k: tuple(sorted(t for t in T if T[t]["core"] == k)) for k in CORES}
    tim = {t: (x["C"], x["T"], x["crit"], x["can"]) for t, x in T.items()}
    mem = {t: (x["r"], x["sram"]) for t, x in T.items()}
    tp = {t: (tim[t], mem[t]) for t in T}
    if file_level:
        a["alloc"] = tuple(sorted(rows.items())); a["tp"] = tuple(sorted(tp.items()))
        a["part"] = tuple(sorted(s["parts"].items()))
    else:
        # entry-level keys; every table also gets an index key so that adding or removing
        # an entry is visible to scopes that cover the whole table
        for k in CORES: a[f"alloc.{k}"] = rows[k]
        for t in T: a[f"tim.{t}"] = tim[t]; a[f"mem.{t}"] = mem[t]
        for t, p in s["parts"].items(): a[f"part.{t}"] = p
        a["part.index"] = tuple(sorted(s["parts"]))
    for t, x in T.items(): a[f"code.{t}"] = x["code"]
    a["ic"] = (s["ic"]["L"], s["ic"]["app_port"])
    a["arch.sec"] = (tuple(sorted(s["controls"])), tuple(sorted(t for t in T if T[t]["can"])))
    a["can"] = tuple(sorted(s["can"].items()))
    a["code.e2e"] = s["e2e_code"]
    a["rig"] = 0; a["item"] = s["item"]
    isl = [t for t in T if T[t]["core"] in ISLAND]
    a["img.island"] = hsh((rows["I0"], rows["I1"], [(tp[t], T[t]["code"]) for t in sorted(isl)],
                           sorted(s["parts"].items()), a["ic"]))
    a["dma"] = s.get("dma", False)
    a["img.app"] = hsh(([rows[k] for k in APP], [(tp[t], T[t]["code"]) for t in sorted(T) if t not in isl], a["dma"]))
    return {k: hsh(v) for k, v in a.items()}

def keys(s, file_level, kind, arg=None):
    """Artifact keys of a kind; file_level maps entry keys to their file."""
    T = s["tasks"]
    if kind == "alloc":
        return ["alloc"] if file_level else [f"alloc.{k}" for k in ([arg] if arg else CORES)]
    if kind in ("tim", "mem", "tp"):
        ts = [t for t in T if (arg is None or T[t]["core"] == arg)]
        if file_level:
            return ["tp"]
        sub = ["tim", "mem"] if kind == "tp" else [kind]
        return [f"{x}.{t}" for t in ts for x in sub]
    if kind == "part":
        return ["part"] if file_level else [f"part.{t}" for t in s["parts"]] + ["part.index"]
    raise KeyError(kind)

def evidence(s, int_variant, file_level=False):
    """Evidence items with manifest M (what the producing tool reads), template-required and
    declared extra scope X, and consumed evidence D."""
    T = s["tasks"]; ev = {}
    K = lambda kind, arg=None: keys(s, file_level, kind, arg)
    ports = CORES if s["ic"]["app_port"] else ISLAND   # cores wired to the SRAM port
    interf_tmpl = [x for k in ports for x in K("alloc", k) + K("tp", k)] + ["ic"]
    if int_variant == "analytic":
        ev["INT"] = dict(type="interference.analytic", M=interf_tmpl + ["dma"], X=[], D=[])
    else:
        ev["INT"] = dict(type="interference.measured", M=["img.island", "img.app", "rig"], X=interf_tmpl, D=[])
    for k in CORES:
        ev[f"RTA:{k}"] = dict(type="rta", M=K("alloc", k) + K("tp", k) + ["report.INT"], X=[], D=["INT"])
    ev["MPU"] = dict(type="mpu", M=K("part"), X=[], D=[])
    ev["HIL"] = dict(type="hil", M=["img.island", "rig"], X=interf_tmpl, D=[])
    for t in T:
        ev[f"UT:{t}"] = dict(type="unit", M=[f"code.{t}"], X=[], D=[])
        ev[f"SA:{t}"] = dict(type="static", M=[f"code.{t}"], X=[], D=[])
    ev["E2E"] = dict(type="e2e", M=["code.e2e", "can"], X=[], D=[])
    ev["TARA"] = dict(type="tara", M=[], X=["arch.sec", "can"], D=[])
    ev["HARA"] = dict(type="review", M=[], X=["item"], D=[])
    for e in ev:
        ev[e]["M"] = sorted(set(ev[e]["M"])); ev[e]["X"] = sorted(set(ev[e]["X"]) - set(ev[e]["M"]))
    return ev

def template(s, etype, file_level=False):
    K = lambda kind, arg=None: keys(s, file_level, kind, arg)
    ports = CORES if s["ic"]["app_port"] else ISLAND
    it = set(x for k in ports for x in K("alloc", k) + K("tp", k)) | {"ic"}
    return {"interference.measured": it, "interference.analytic": it, "hil": it,
            "tara": {"arch.sec"}, "review": {"item"}}.get(etype, set())

def trace_links(s):
    tl = {f"UT:{t}": [f"code.{t}"] for t in s["tasks"]}
    tl.update({f"SA:{t}": [f"code.{t}"] for t in s["tasks"]})
    tl["HIL"] = ["img.island"]; tl["E2E"] = ["code.e2e"]
    return tl

def claims(s):
    c = {f"TIM:{k}": [f"RTA:{k}", "INT"] + (["HIL"] if k == "I0" else []) for k in CORES}
    c["FFI"] = ["INT", "MPU"]
    c["SEC"] = ["TARA", "E2E"]
    for t in s["tasks"]:
        c[f"COR:{t}"] = [f"UT:{t}", f"SA:{t}"]
    c["TOP"] = ["HARA"]
    return c

# ----------------------------------------------------------------------------- changes
def change(s, kind, rng):
    s = copy.deepcopy(s); T = s["tasks"]
    isl = [t for t in T if T[t]["core"] in ISLAND and t not in ("BrakeCtl",)]
    app = [t for t in T if T[t]["core"] in APP]
    if kind == "realloc_island":
        t = rng.choice(isl); T[t]["core"] = "I0" if T[t]["core"] == "I1" else "I1"
        T[t]["crit"] = "D" if T[t]["core"] == "I0" else "B"
    elif kind in ("add_ids_island", "add_ids_app"):
        core = "I1" if kind == "add_ids_island" else rng.choice(APP)
        T["IDS"] = dict(core=core, C=round(rng.uniform(100, 300), 1), T=10000, r=2.5,
                        sram=True, crit="QM", code=0, behav=0, can=True)
        if core in ISLAND:
            end = max(b + z for b, z in s["parts"].values())
            s["parts"]["IDS"] = (end + 1024, 8192)
        s["controls"] = sorted(set(s["controls"]) | {"IDS"})
    elif kind == "realloc_app":
        t = rng.choice(app); T[t]["core"] = rng.choice([k for k in APP if k != T[t]["core"]])
    elif kind == "app_sram_enable":
        cand = [t for t in app if not T[t]["sram"]] or app
        t = rng.choice(cand); T[t]["sram"] = True
    elif kind == "wcet_change":
        t = rng.choice(isl + ["BrakeCtl"]); T[t]["C"] = round(T[t]["C"] * rng.uniform(1.1, 1.3), 1)
        T[t]["code"] += 1; T[t]["behav"] += 1
    elif kind == "refactor":
        t = rng.choice(isl + ["BrakeCtl"]); T[t]["code"] += 1
    elif kind == "period_change":
        t = rng.choice(isl); T[t]["T"] = rng.choice([p * 1000 for p in PERIODS_MS if p * 1000 != T[t]["T"] and p >= 2])
    elif kind == "ic_change":
        s["ic"]["L"] = round(s["ic"]["L"] * rng.choice([0.5, 1.5, 2.0]), 3)
    elif kind == "partition_resize":
        t = rng.choice(sorted(s["parts"])); b, z = s["parts"][t]; s["parts"][t] = (b, int(z * rng.uniform(1.1, 1.6)))
    elif kind == "can_add_msg":
        s["can"][f"msgN{rng.randint(0, 99)}"] = rng.random() < 0.5
    elif kind == "dma_enable":
        s["dma"] = True
    elif kind == "app_code_change":
        t = rng.choice(app); T[t]["C"] = round(T[t]["C"] * rng.uniform(1.1, 1.5), 1)
        T[t]["code"] += 1; T[t]["behav"] += 1
    return s

CHANGES = ["add_ids_island", "add_ids_app", "realloc_island", "realloc_app", "app_sram_enable",
           "wcet_change", "refactor", "period_change", "ic_change", "partition_resize",
           "can_add_msg", "app_code_change", "dma_enable"]

# ----------------------------------------------------------------------------- methods
def closure(flagged, ev):
    f = set(flagged); changed = True
    while changed:
        changed = False
        for e, d in ev.items():
            if e not in f and set(d["D"]) & f:
                f.add(e); changed = True
    return f

def flagged_sets(s0, s1, int_variant):
    out = {}
    for fl in (False, True):
        a0, a1 = artifacts(s0, fl), artifacts(s1, fl)
        C = {k for k in set(a0) | set(a1) if a0.get(k) != a1.get(k)}
        ev = evidence(s0, int_variant, fl)
        man = {e for e, d in ev.items() if set(d["M"]) & C}
        direct = {e for e, d in ev.items() if (set(d["M"]) | set(d["X"])) & C}
        if not fl:
            tl = {e for e, arts in trace_links(s0).items() if set(arts) & C}
            out["trace"] = tl; out["manifest"] = man
            out["aag_direct"] = direct; out["aag_closed"] = closure(direct, ev)
            out["_ev"] = ev
        else:
            out["aag_closed_file"] = closure(direct, ev)
    return out

def perturb_scopes(ev, s, rng, p_omit, p_add, checks, universe):
    """A1 proposes scopes = true scope with omissions/inventions; with checks, a proposal that
    misses a manifest or template entry is rejected and corrected by a human."""
    new, escal = {}, 0
    for e, d in ev.items():
        true = set(d["M"]) | set(d["X"])
        prop = {a for a in true if rng.random() >= p_omit}
        if rng.random() < p_add:
            prop.add(rng.choice(universe))
        if checks:
            need = set(d["M"]) | (template(s, d["type"]) & true)
            if not need <= prop:
                prop = true | (prop - true); escal += 1   # human fixes the omission
        new[e] = dict(d, S=prop)
    return new, escal

# ----------------------------------------------------------------------------- main
def run(n_sys=300, seed=1):
    rng = random.Random(seed)
    methods = ["trace", "manifest", "aag_direct", "aag_closed", "aag_cutoff", "aag_closed_file"]
    agg = {v: {c: defaultdict(float) for c in CHANGES + ["ALL"]} for v in ("analytic", "measured")}
    scope_err = defaultdict(lambda: defaultdict(float))
    p_grid = [0.0, 0.05, 0.1, 0.2, 0.3]
    for i in range(n_sys):
        s0 = gen_system(rng)
        for ck in CHANGES:
            s1 = change(s0, ck, rng)
            ev_ids = list(evidence(s0, "analytic").keys())
            o0, o1 = outputs(s0, ev_ids), outputs(s1, ev_ids)
            stale = {e for e in ev_ids if o0[e] != o1[e]}
            crit = {e for e in stale if critical(e, o0[e], o1[e])}
            cl = claims(s0)
            for variant in ("analytic", "measured"):
                fs = flagged_sets(s0, s1, variant)
                ev = fs.pop("_ev")
                # early cut-off: re-run upstream items first; a consumer is re-run only if an
                # upstream output actually changed (known once the upstream re-run is done)
                cut = set(fs["aag_direct"]); grow = True
                while grow:
                    grow = False
                    for e, d in ev.items():
                        if e not in cut and any(x in cut and o0[x] != o1[x] for x in d["D"]):
                            cut.add(e); grow = True
                fs["aag_cutoff"] = cut
                for bucket in (ck, "ALL"):
                    A = agg[variant][bucket]
                    A["n"] += 1; A["stale"] += len(stale); A["crit"] += len(crit)
                    A["any_stale"] += bool(stale)
                    A["need_h"] += sum(COST_H[ev[e]["type"]] for e in stale)
                    for m in methods:
                        f = fs[m]
                        A[f"{m}.tp"] += len(f & stale); A[f"{m}.fp"] += len(f - stale)
                        A[f"{m}.miss"] += len(stale - f); A[f"{m}.critmiss"] += len(crit - f)
                        A[f"{m}.scen_miss"] += bool(stale - f); A[f"{m}.scen_critmiss"] += bool(crit - f)
                        A[f"{m}.hours"] += sum(COST_H[ev[e]["type"]] for e in f if e in ev)
                        if bucket == "ALL":
                            for e in stale:
                                ty = ev[e]["type"].split(".")[0]
                                A[f"type.{ty}.stale"] += 1 if m == "trace" else 0
                                A[f"{m}.type.{ty}.hit"] += e in f
                        bad = [c for c, es in cl.items() if set(es) & stale and not set(es) & f]
                        A[f"{m}.false_sup"] += len(bad)
                        A[f"{m}.scen_false_sup"] += bool(bad)
            # scope-error sensitivity (measured interference variant: largest non-manifest scopes)
            ev = evidence(s0, "measured")
            a0, a1 = artifacts(s0), artifacts(s1)
            C = {k for k in set(a0) | set(a1) if a0.get(k) != a1.get(k)}
            universe = sorted(a0)
            for p in p_grid:
                for checks in (False, True):
                    pr, esc = perturb_scopes(ev, s0, rng, p, 0.05, checks, universe)
                    f = closure({e for e, d in pr.items() if d["S"] & C}, ev)
                    k = f"p={p}|checks={checks}"
                    S = scope_err[k]
                    S["n"] += 1; S["miss"] += len(stale - f); S["stale"] += len(stale)
                    S["critmiss"] += len(crit - f); S["scen_miss"] += bool(stale - f)
                    S["escalations"] += esc; S["n_ev"] += len(ev)
                    S["fp"] += len(f - stale)
    return agg, scope_err

def summarize(agg, scope_err):
    out = {"methods": {}, "scope_errors": {}}
    for variant, by in agg.items():
        out["methods"][variant] = {}
        for ck, A in by.items():
            row = {"scenarios": int(A["n"]), "stale_items": int(A["stale"]), "critical_items": int(A["crit"]),
                   "scenarios_with_stale": int(A["any_stale"]), "needed_hours": round(A["need_h"], 1)}
            for m in ["trace", "manifest", "aag_direct", "aag_closed", "aag_cutoff", "aag_closed_file"]:
                tp, fp, miss = A[f"{m}.tp"], A[f"{m}.fp"], A[f"{m}.miss"]
                row[m] = {"recall": round(tp / (tp + miss), 4) if tp + miss else None,
                          "precision": round(tp / (tp + fp), 4) if tp + fp else None,
                          "missed_items": int(miss), "missed_critical": int(A[f"{m}.critmiss"]),
                          "scenarios_with_miss": int(A[f"{m}.scen_miss"]),
                          "scenarios_with_critical_miss": int(A[f"{m}.scen_critmiss"]),
                          "claims_falsely_supported": int(A[f"{m}.false_sup"]),
                          "scenarios_with_false_support": int(A[f"{m}.scen_false_sup"]),
                          "reverification_hours": round(A[f"{m}.hours"], 1)}
            if ck == "ALL":
                types = sorted({k.split(".")[1] for k in A if k.startswith("type.")})
                row["per_type"] = {ty: {"stale": int(A[f"type.{ty}.stale"]),
                                        **{m: round(A[f"{m}.type.{ty}.hit"] / A[f"type.{ty}.stale"], 4)
                                           for m in ["trace", "manifest", "aag_direct", "aag_closed", "aag_cutoff"]}}
                                   for ty in types}
            out["methods"][variant][ck] = row
    for k, S in sorted(scope_err.items()):
        out["scope_errors"][k] = {"recall": round(1 - S["miss"] / S["stale"], 4) if S["stale"] else None,
                                  "missed_items": int(S["miss"]), "missed_critical": int(S["critmiss"]),
                                  "scenarios_with_miss": int(S["scen_miss"]), "scenarios": int(S["n"]),
                                  "escalation_rate": round(S["escalations"] / S["n_ev"], 4),
                                  "false_positives": int(S["fp"])}
    return out

if __name__ == "__main__":
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 300
    seed = int(sys.argv[2]) if len(sys.argv) > 2 else 1
    agg, se = run(n, seed)
    res = summarize(agg, se); res["config"] = {"n_systems": n, "seed": seed, "changes": CHANGES, "cost_hours": COST_H}
    json.dump(res, sys.stdout, indent=1)

In [ ]:
%%writefile /content/aag/a1_eval.py
#!/usr/bin/env python3
"""Evaluate agent A1 (evidence ingestion): from a tool report, propose the evidence scope S(e)
and consumed evidence D(e).

Reports are generated from benchmark systems (bench.py), so the true scope is known. Each
evidence item is rendered in one of three styles:
  structured : tool-native output that names artifact keys literally;
  vendor     : a different tool's output using aliases (core and task names differ from keys);
  prose      : an engineer's notes / review minutes in natural language.
Proposers:
  parser            : regex for literal artifact keys and evidence ids (no model);
  parser+templates  : parser output plus the type template (deterministic, no model);
  claude:<model>    : Anthropic API (needs ANTHROPIC_API_KEY);
  openai:<model>    : any OpenAI-compatible endpoint, e.g. a local open-weight model
                      (needs OPENAI_BASE_URL, optional OPENAI_API_KEY).
Usage:
  python3 a1_eval.py gen  N_SYSTEMS SEED out.jsonl         # generate tasks
  python3 a1_eval.py run  tasks.jsonl PROPOSER out.jsonl    # propose scopes
  python3 a1_eval.py score tasks.jsonl proposals.jsonl      # per-type error rates (JSON)
"""
import json, os, random, re, sys, time, urllib.request
import bench

STYLES = ["structured", "vendor", "prose"]

# ----------------------------------------------------------------------------- catalogue
def catalogue(s):
    """Artifact keys of a system with human-readable descriptions (what A1 sees)."""
    T = s["tasks"]; cat = {}
    for k in bench.CORES:
        cat[f"alloc.{k}"] = f"allocation row of core {k} ({'safety island' if k in bench.ISLAND else 'application cluster'})"
    for t, x in sorted(T.items()):
        cat[f"tim.{t}"] = f"timing parameters (WCET, period) of task {t}"
        cat[f"mem.{t}"] = f"memory-access parameters (SRAM use, access rate) of task {t}"
        cat[f"code.{t}"] = f"source code unit of task {t}"
    for t in sorted(s["parts"]):
        cat[f"part.{t}"] = f"MPU partition entry of task {t}"
    cat["part.index"] = "index of the MPU partition table (set of entries)"
    cat.update({"ic": "interconnect configuration (SRAM port arbitration, slot latency, which clusters are wired to the port)",
                "dma": "application-cluster DMA engine configuration",
                "arch.sec": "security-control architecture view (controls, tasks with CAN access)",
                "can": "CAN communication matrix (messages, E2E protection)",
                "code.e2e": "end-to-end protection library code", "rig": "HIL test rig configuration",
                "item": "item definition", "img.island": "built software image of the safety island",
                "img.app": "built software image of the application cluster",
                "report.INT": "interference analysis report file"})
    return cat

def aliases(rng):
    core = {"I0": "SafetyCore0", "I1": "SafetyCore1", "A0": "AppCPU0", "A1": "AppCPU1", "A2": "AppCPU2", "A3": "AppCPU3"}
    return core

def tname(t, style, core_alias):
    if style != "vendor":
        return t
    if t in ("BrakeCtl", "WSpeed", "IDS"):
        return {"BrakeCtl": "Brake_Control_5ms", "WSpeed": "WheelSpeed_Plaus_5ms", "IDS": "CAN_IDS"}[t]
    core, _, i = t.partition("t")
    return f"{core_alias[core]}_Runnable{i}"

# ----------------------------------------------------------------------------- reports
def render(s, e, d, style, rng):
    T = s["tasks"]; ca = aliases(rng); n = lambda t: tname(t, style, ca)
    kind, _, arg = e.partition(":")
    port_cores = bench.CORES if s["ic"]["app_port"] else bench.ISLAND
    L = s["ic"]["L"]
    if kind == "RTA":
        ts = sorted([t for t in T if T[t]["core"] == arg], key=lambda t: T[t]["T"])
        if style == "structured":
            rows = "\n".join(f"  {t}: C={T[t]['C']}us T={T[t]['T']}us inputs=[tim.{t}, mem.{t}]" for t in ts)
            return f"RTA-Tool v3.2 report\ncore: {arg}\ninput allocation: alloc.{arg}\ninterference input: report.INT (from evidence INT)\ntasks:\n{rows}\nresult: all deadlines met"
        if style == "vendor":
            rows = "\n".join(f"| {n(t)} | {T[t]['C']} | {T[t]['T']} | {T[t]['r']} acc/us |" for t in ts)
            return f"SchedAnalyzer Pro - Response Time Summary\nTarget: {ca[arg]}\nMulticore interference bounds imported from: interference_bounds.xml\n| Runnable | WCET(us) | Period(us) | Mem.intensity |\n{rows}"
        return (f"Notes from the timing review for {ca[arg]}: we re-ran the response-time analysis for the {len(ts)} "
                f"tasks mapped to that core ({', '.join(n(t) for t in ts)}), using the execution budgets and memory profiles from "
                f"the task configuration and the inflated execution times coming out of last week's interference analysis.")
    if kind == "INT":
        if style == "structured":
            return (f"IntfMeasure v1.4 log\nimages: img.island, img.app\nrig: rig\nmeasured worst per-access delay on island cores; "
                    f"co-runner load generated on {', '.join(port_cores)}")
        if style == "vendor":
            return (f"Multicore Interference Campaign Report\nBuilds under test: island-fw-build-{rng.randint(100,999)}, app-fw-build-{rng.randint(100,999)}\n"
                    f"Bench: HIL-B2\nScenario: all co-runner contenders active on {', '.join(ca[c] for c in port_cores)}; "
                    f"SRAM port slot setting {int(L*1000)} ns")
        return ("We measured the shared-SRAM interference on target with every other core running its normal "
                "load plus stress kernels. The bound holds for the current task-to-core mapping and the current interconnect arbitration settings.")
    if kind == "MPU":
        ents = sorted(s["parts"])
        if style == "structured":
            return "MPU-Check report\nentries checked: " + ", ".join(f"part.{t}" for t in ents) + "\nindex: part.index\nresult: no overlap"
        if style == "vendor":
            return "MemProt Verifier\nRegions: " + "; ".join(f"{n(t)}@0x{s['parts'][t][0]:06x}+{s['parts'][t][1]}" for t in ents)
        return f"Reviewed the MPU region table ({len(ents)} regions) for overlaps between partitions of different criticality; none found."
    if kind == "HIL":
        if style == "structured":
            return "HIL TC-17 log\nimage: img.island\nrig: rig\nmeasured brake latency within budget"
        if style == "vendor":
            return f"TestStand run TC-17 Brake_Latency\nDUT build island-fw-build-{rng.randint(100,999)} on HIL-B2\nmax latency 2.41 ms (n=500)"
        return ("Ran the brake pedal-to-command latency test on the rig with the current island firmware; the application cluster was "
                "running its production load at the same time, so the result reflects shared-memory contention from the other cores.")
    if kind in ("UT", "SA"):
        what = "unit tests" if kind == "UT" else "static analysis"
        if style == "structured":
            return f"{what} report\nunit: code.{arg}\nresult: pass"
        if style == "vendor":
            return f"{'VectorCAST' if kind == 'UT' else 'Polyspace'} summary for module {n(arg)}: 0 failures"
        return f"Completed {what} of the {n(arg)} software component; all findings dispositioned."
    if kind == "E2E":
        msgs = [m for m, p in sorted(s["can"].items()) if p]
        if style == "structured":
            return "E2E test report\nlibrary: code.e2e\nmatrix: can\nmessages: " + ", ".join(msgs)
        if style == "vendor":
            return "E2E Profile 5 conformance: " + ", ".join(msgs) + " -- PASS (lib E2E_P05 v2.1)"
        return "Checked end-to-end protection for all protected messages in the CAN database using the E2E library; all pass."
    if kind == "TARA":
        rows = "; ".join(m for m in sorted(s["can"]))
        if style == "structured":
            return f"TARA sheet\narchitecture: arch.sec\nmessages: can ({rows})\ncontrols: {', '.join(s['controls'])}"
        if style == "vendor":
            return f"Threat analysis workbook rev C. Assets: {rows}. Controls in place: {', '.join(s['controls'])}."
        return ("Threat analysis workshop for the CAN interface: we went through each message in the communication matrix "
                "and rated attack feasibility against the security controls shown in the current security architecture.")
    if kind == "HARA":
        return {"structured": "HARA review\nitem: item", "vendor": "Hazard analysis review minutes, item definition rev 4",
                "prose": "Reviewed the hazard analysis against the current item definition."}[style]
    raise KeyError(e)

def truth(d):
    return sorted(set(d["M"]) | set(d["X"]))

def gen(n_sys, seed, out):
    rng = random.Random(seed)
    with open(out, "w") as f:
        for i in range(n_sys):
            s = bench.gen_system(rng)
            ev = bench.evidence(s, "measured")
            # one item per evidence type per system keeps the mix balanced
            picks = ["INT", "MPU", "HIL", "E2E", "TARA", "HARA", f"RTA:{rng.choice(bench.CORES)}",
                     f"UT:{rng.choice(sorted(s['tasks']))}", f"SA:{rng.choice(sorted(s['tasks']))}"]
            cat = catalogue(s)
            for e in picks:
                style = rng.choice(STYLES)
                d = ev[e]
                f.write(json.dumps({"id": f"s{i}:{e}", "system": i, "seed": seed, "evidence": e, "type": d["type"],
                                    "style": style, "report": render(s, e, d, style, rng),
                                    "catalogue": cat, "evidence_ids": sorted(ev), "truth_scope": truth(d),
                                    "truth_consumes": d["D"], "manifest": d["M"],
                                    "template": sorted(bench.template(s, d["type"]))}) + "\n")

# ----------------------------------------------------------------------------- proposers
PROMPT = """You maintain an assurance graph for a safety-critical multicore ECU. A tool report or analysis record is given below.
Propose its evidence scope: every configuration artifact (from the catalogue) whose change could change this result, and every
other evidence item whose result this one consumes. Include dependencies that the report does not name explicitly if the kind of
analysis depends on them (e.g., multicore interference depends on what runs on the other cores). Do not include artifacts the
result cannot depend on.

Evidence id: {eid}
Report:
<<<
{report}
>>>

Artifact catalogue (key: description):
{cat}

Other evidence ids: {evs}

Answer with JSON only: {{"scope": [artifact keys], "consumes": [evidence ids]}}"""

def parser(task):
    txt = task["report"]
    keys = [k for k in task["catalogue"] if re.search(r"(?<![\w.])" + re.escape(k) + r"(?![\w])", txt)]
    cons = [e for e in task["evidence_ids"] if re.search(r"(?<![\w:])" + re.escape(e) + r"(?![\w:])", txt) and e != task["evidence"]]
    return {"scope": keys, "consumes": cons}

def call_claude(model, prompt):
    req = urllib.request.Request("https://api.anthropic.com/v1/messages", method="POST",
        headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01", "content-type": "application/json"},
        data=json.dumps({"model": model, "max_tokens": 2000, "temperature": 0,
                         "messages": [{"role": "user", "content": prompt}]}).encode())
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    return "".join(b.get("text", "") for b in out["content"]), out.get("usage", {})

def call_openai(model, prompt):
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    req = urllib.request.Request(base + "/chat/completions", method="POST",
        headers={"content-type": "application/json", "authorization": "Bearer " + os.environ.get("OPENAI_API_KEY", "none")},
        data=json.dumps({"model": model, "temperature": 0, "messages": [{"role": "user", "content": prompt}]}).encode())
    with urllib.request.urlopen(req, timeout=300) as r:
        out = json.load(r)
    return out["choices"][0]["message"]["content"], out.get("usage", {})

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        j = json.loads(m.group(0)) if m else {}
    except Exception:
        j = {}
    return {"scope": [str(x) for x in j.get("scope", [])], "consumes": [str(x) for x in j.get("consumes", [])]}

def run(tasks_path, proposer, out_path):
    done = set()
    if os.path.exists(out_path):
        done = {json.loads(l)["id"] for l in open(out_path)}
    with open(out_path, "a") as f:
        for line in open(tasks_path):
            t = json.loads(line)
            if t["id"] in done:
                continue
            usage, raw = {}, None
            if proposer == "parser":
                p = parser(t)
            elif proposer == "parser+templates":
                p = parser(t); p["scope"] = sorted(set(p["scope"]) | set(t["template"]))
                if t["type"] == "rta": p["consumes"] = sorted(set(p["consumes"]) | {"INT"})
            else:
                backend, _, model = proposer.partition(":")
                prompt = PROMPT.format(eid=t["evidence"], report=t["report"],
                                       cat="\n".join(f"{k}: {v}" for k, v in t["catalogue"].items()),
                                       evs=", ".join(e for e in t["evidence_ids"] if e != t["evidence"]))
                for attempt in range(5):
                    try:
                        raw, usage = (call_claude if backend == "claude" else call_openai)(model, prompt)
                        break
                    except Exception as ex:
                        time.sleep(2 ** attempt); raw = f"ERROR {ex}"
                p = parse_json(raw or "")
            f.write(json.dumps({"id": t["id"], "proposer": proposer, **p, "usage": usage, "raw": raw}) + "\n"); f.flush()

# ----------------------------------------------------------------------------- scoring
def score(tasks_path, prop_path):
    tasks = {json.loads(l)["id"]: json.loads(l) for l in open(tasks_path)}
    by = {}
    for line in open(prop_path):
        p = json.loads(line); t = tasks[p["id"]]
        true, prop = set(t["truth_scope"]), set(p["scope"])
        tc, pc = set(t["truth_consumes"]), set(p["consumes"])
        checked_ok = set(t["manifest"]) <= prop and set(t["template"]) <= prop and \
                     (t["type"] != "rta" or "INT" in pc)
        for key in (t["type"], f"style:{t['style']}", "ALL"):
            b = by.setdefault(key, dict(n=0, true=0, omitted=0, invented=0, items_with_omission=0,
                                        consume_missed=0, rejected_by_checks=0, omission_after_checks=0))
            b["n"] += 1; b["true"] += len(true); b["omitted"] += len(true - prop); b["invented"] += len(prop - true)
            b["items_with_omission"] += bool(true - prop) or bool(tc - pc)
            b["consume_missed"] += len(tc - pc)
            b["rejected_by_checks"] += not checked_ok
            b["omission_after_checks"] += checked_ok and (bool(true - prop) or bool(tc - pc))
    for b in by.values():
        b["omission_rate"] = round(b["omitted"] / b["true"], 4) if b["true"] else None
        b["items_with_omission_rate"] = round(b["items_with_omission"] / b["n"], 4)
        b["escalation_rate"] = round(b["rejected_by_checks"] / b["n"], 4)
        b["undetected_omission_rate"] = round(b["omission_after_checks"] / b["n"], 4)
    return by

if __name__ == "__main__":
    cmd = sys.argv[1]
    if cmd == "gen":
        gen(int(sys.argv[2]), int(sys.argv[3]), sys.argv[4])
    elif cmd == "run":
        run(sys.argv[2], sys.argv[3], sys.argv[4])
    elif cmd == "score":
        json.dump(score(sys.argv[2], sys.argv[3]), sys.stdout, indent=1)

In [ ]:
%%writefile /content/aag/agents.py
#!/usr/bin/env python3
"""Language-model agents for the Agentic Assurance Graph (AAG), and their evaluation.

Agents (Sec. "Bounded Agent Roles" of the paper) propose; the deterministic core decides.
  A1  evidence ingestion: reads a tool report or analysis record and proposes the evidence
      scope S(e) (artifact keys) and the evidence items it consumes D(e), with a rationale.
  A3  re-verification planning: reads an engineering change request, the graph and the core's
      invalidation set I(C); proposes an order, a causal rationale per item, and any extra items
      it believes need re-verification (extras can only add work, never remove it).
  IMPACT (baseline, not an AAG role): the same model is asked to decide on its own which
      evidence a change makes stale, with no core. This measures what would slip through if
      the model's judgement were trusted.

Scenarios are a sample of the 19,500 benchmark scenarios: the first K systems of bench.run for
seeds 1-3 (systems are i.i.d., so the first K are a random sample), with the exact changed
systems that bench.run produced. Ground truth comes from re-running the analyses (bench.outputs).

Everything except the model call is deterministic and runs offline. Offline proposers
("oracle", "parser+templates", "empty") exercise the full pipeline without a model.

Usage (offline):
  python3 agents.py tasks OUT_DIR K1 K3            # write a1_tasks.jsonl, a3_tasks.jsonl
  python3 agents.py offline OUT_DIR PROPOSER       # answer all tasks without a model
  python3 agents.py score OUT_DIR ANSWERS.jsonl    # write score JSON
The Colab notebook (../notebooks/run_agents_colab.ipynb) calls the same functions and sends the
prompts to the Anthropic Message Batches API.
"""
import copy, json, os, random, re, sys
from collections import defaultdict
import bench
import a1_eval

MODEL = "claude-opus-5-5"          # pinned for reproducibility
EFFORT = "medium"                  # explicit (also the model's default)
PRICE_IN, PRICE_OUT = 4.0, 20.0    # USD per million tokens, standard; batch is 50%
BATCH_DISCOUNT = 0.5
MODEL_ITEMS = ("INT", "RTA", "MPU", "HIL", "E2E", "TARA", "HARA")   # A1 runs on these kinds
INT_VARIANT = "measured"           # largest non-manifest scopes (as in RQ4)

# ============================================================================ scenarios
def sample_scenarios(k, seeds=(1, 2, 3)):
    """First k systems of bench.run(., seed) for each seed, with the 13 changed systems that
    bench.run itself generated (same random stream), i.e. exact benchmark scenarios."""
    out = []
    for seed in seeds:
        rec = []
        orig = bench.change
        def spy(s, kind, rng, _o=orig):
            s1 = _o(s, kind, rng); rec.append((copy.deepcopy(s), kind, copy.deepcopy(s1))); return s1
        bench.change = spy
        try:
            bench.run(k, seed)
        finally:
            bench.change = orig
        for j in range(k):
            chunk = rec[j * len(bench.CHANGES):(j + 1) * len(bench.CHANGES)]
            out.append(dict(sid=f"seed{seed}.sys{j}", seed=seed, idx=j, s0=chunk[0][0],
                            changes=[(ck, s1) for _, ck, s1 in chunk]))
    return out

def truth_for(s0, s1):
    ev_ids = list(bench.evidence(s0, INT_VARIANT).keys())
    o0, o1 = bench.outputs(s0, ev_ids), bench.outputs(s1, ev_ids)
    stale = {e for e in ev_ids if o0[e] != o1[e]}
    crit = {e for e in stale if bench.critical(e, o0[e], o1[e])}
    a0, a1 = bench.artifacts(s0), bench.artifacts(s1)
    C = {k for k in set(a0) | set(a1) if a0.get(k) != a1.get(k)}
    return stale, crit, C

def core_flag(ev, C, scopes=None, consumes=None):
    """AAG core: direct invalidation by scope intersection, closed over consumes."""
    S = scopes or {e: set(d["M"]) | set(d["X"]) for e, d in ev.items()}
    D = consumes or {e: set(d["D"]) for e, d in ev.items()}
    f = {e for e in ev if S[e] & C}
    grow = True
    while grow:
        grow = False
        for e in ev:
            if e not in f and D[e] & f:
                f.add(e); grow = True
    return f

def check_proposal(manifest, true_scope, etype, template, scope, consumes):
    """Deterministic scope checks (Sec. Scope Completeness): manifest containment, type
    template containment, and the consumes edge every RTA has to the interference result.
    Returns the list of violated requirements (empty = accepted)."""
    need = set(manifest) | (set(template) & set(true_scope))
    v = sorted(need - set(scope))
    if etype == "rta" and "INT" not in consumes:
        v.append("consumes:INT")
    return v

# ============================================================================ text rendering
def describe_change(s0, s1, ck):
    """An engineering change request (ECR) as an engineer would write it. Uses artifact-level
    names (task and core ids) but never lists which evidence is affected."""
    T0, T1 = s0["tasks"], s1["tasks"]
    moved = [t for t in T0 if t in T1 and T0[t]["core"] != T1[t]["core"]]
    if ck in ("add_ids_island", "add_ids_app"):
        x = T1["IDS"]
        txt = (f"Add a CAN intrusion-detection task IDS on core {x['core']}: period 10 ms, WCET {x['C']} us, "
               f"reads every CAN frame, keeps its signature tables in the shared island SRAM. "
               f"Security architecture updated to list IDS as a control.")
        if "IDS" in s1["parts"]:
            txt += " A new MPU partition is created for IDS."
        return txt
    if ck in ("realloc_island", "realloc_app"):
        t = moved[0]
        return f"Load balancing: move task {t} from core {T0[t]['core']} to core {T1[t]['core']}. No code change."
    if ck == "app_sram_enable":
        t = [t for t in T0 if T0[t]["sram"] != T1[t]["sram"]]
        t = t[0] if t else sorted(t for t in T0 if T0[t]["core"] in bench.APP)[0]
        return f"Task {t} (core {T0[t]['core']}) moves its working buffers from local RAM to the shared island SRAM. No code change."
    if ck in ("wcet_change", "app_code_change"):
        t = [t for t in T0 if T0[t]["C"] != T1[t]["C"]][0]
        return (f"Feature change in task {t} (core {T0[t]['core']}): new filtering logic; source code and unit "
                f"behaviour change; measured WCET grows from {T0[t]['C']} us to {T1[t]['C']} us.")
    if ck == "refactor":
        t = [t for t in T0 if T0[t]["code"] != T1[t]["code"]][0]
        return f"Refactoring of task {t} (core {T0[t]['core']}): source restructured, behaviour and WCET unchanged."
    if ck == "period_change":
        t = [t for t in T0 if T0[t]["T"] != T1[t]["T"]][0]
        return f"Period of task {t} (core {T0[t]['core']}) changes from {T0[t]['T']//1000} ms to {T1[t]['T']//1000} ms."
    if ck == "ic_change":
        return (f"Interconnect configuration: SRAM port arbitration slot latency changes from "
                f"{int(s0['ic']['L']*1000)} ns to {int(s1['ic']['L']*1000)} ns.")
    if ck == "partition_resize":
        t = [t for t in s0["parts"] if s0["parts"][t] != s1["parts"][t]][0]
        return f"MPU partition of task {t} grows from {s0['parts'][t][1]} to {s1['parts'][t][1]} bytes (same base address)."
    if ck == "can_add_msg":
        m = [m for m in s1["can"] if m not in s0["can"]]
        m = m[0] if m else sorted(s1["can"])[-1]
        return f"Add CAN message {m} to the communication matrix ({'E2E protected' if s1['can'][m] else 'not E2E protected'})."
    if ck == "dma_enable":
        return "Enable the application-cluster DMA engine to copy camera frames into a staging buffer in the shared island SRAM."
    raise KeyError(ck)

def render_graph(s, ev):
    lines = []
    for e in sorted(ev):
        d = ev[e]
        sc = sorted(set(d["M"]) | set(d["X"]))
        lines.append(f"{e} [{d['type']}] scope: {', '.join(sc) or '-'}" + (f"; consumes: {', '.join(d['D'])}" if d["D"] else ""))
    return "\n".join(lines)

def render_catalogue(s):
    return "\n".join(f"{k}: {v}" for k, v in a1_eval.catalogue(s).items())

# ============================================================================ prompts + schemas
SYSTEM = ("You are an assistant inside an assurance-maintenance tool for a safety-critical multicore "
          "automotive ECU (two lockstep safety-island cores I0, I1 and four application cores A0-A3 sharing an "
          "island SRAM port). Your output is a proposal that deterministic checks and a human engineer will review; "
          "you never decide whether a claim holds. Be complete and precise. Answer only in the requested JSON.")

A1_PROMPT = """Task: evidence ingestion. A tool report or analysis record is given below. Propose its evidence scope:
every configuration artifact (from the catalogue) whose change could change this result, and every other evidence
item whose result this one consumes. Include dependencies that the report does not name explicitly when the kind of
analysis depends on them (for example, multicore interference depends on what runs on the other cores that reach the
shared SRAM port). Do not include artifacts the result cannot depend on. Give a short rationale.

Evidence id: {eid}
Report:
<<<
{report}
>>>

Artifact catalogue (key: description):
{cat}

Other evidence ids: {evs}"""

A1_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["scope", "consumes", "rationale"],
             "properties": {"scope": {"type": "array", "items": {"type": "string"}},
                            "consumes": {"type": "array", "items": {"type": "string"}},
                            "rationale": {"type": "string"}}}

IMPACT_PROMPT = """Task: change-impact analysis. An engineering change request (ECR) is given, together with the assurance
graph: every evidence item with its type, its scope (the artifacts its result depends on) and the evidence results it
consumes. Decide which evidence items this change makes stale, that is, whose result could differ if the analysis or
test were re-run on the changed system. For each, give the reason in one sentence.

ECR: {ecr}

Artifact catalogue (key: description):
{cat}

Assurance graph:
{graph}"""

IMPACT_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["stale"],
                 "properties": {"stale": {"type": "array", "items": {
                     "type": "object", "additionalProperties": False, "required": ["evidence", "reason"],
                     "properties": {"evidence": {"type": "string"}, "reason": {"type": "string"}}}}}}

A3_PROMPT = """Task: re-verification planning. An engineering change request (ECR) is given, together with the assurance
graph and the invalidation set I(C) that the deterministic core computed (items whose scope contains a changed
artifact, closed over consumed results). Propose a re-verification plan:
- "order": every item of I(C) exactly once, in the order to re-run them. An item must come after every item of I(C)
  it consumes. Among independent items, put cheap ones first.
- "items": for each item of I(C), the causal path from the change to it, as a list that starts with one changed artifact
  key from the item's scope, or with an item of I(C) that this item consumes, and ends with the item id; plus a
  one-sentence rationale.
- "extra": evidence items NOT in I(C) that you believe the change could still make stale (for example through a
  dependency no scope records), each with a reason. Leave it empty if there are none. Extras only add work.

Changed artifact keys C (from content hashes): {changed}
I(C): {inv}

ECR: {ecr}

Artifact catalogue (key: description):
{cat}

Assurance graph:
{graph}"""

A3_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["order", "items", "extra"],
             "properties": {
                 "order": {"type": "array", "items": {"type": "string"}},
                 "items": {"type": "array", "items": {"type": "object", "additionalProperties": False,
                           "required": ["evidence", "path", "rationale"],
                           "properties": {"evidence": {"type": "string"},
                                          "path": {"type": "array", "items": {"type": "string"}},
                                          "rationale": {"type": "string"}}}},
                 "extra": {"type": "array", "items": {"type": "object", "additionalProperties": False,
                           "required": ["evidence", "reason"],
                           "properties": {"evidence": {"type": "string"}, "reason": {"type": "string"}}}}}}

SCHEMAS = {"a1": A1_SCHEMA, "impact": IMPACT_SCHEMA, "a3": A3_SCHEMA}
MAX_TOKENS = {"a1": 16000, "impact": 24000, "a3": 32000}

# ============================================================================ task generation
def gen_tasks(k1, k3, seed=7):
    """A1 tasks on the first k1 systems per seed, IMPACT and A3 tasks on the first k3 systems
    per seed (all 13 changes). Report styles are drawn with a fixed seed."""
    rng = random.Random(seed)
    scen = sample_scenarios(max(k1, k3))
    a1, a3 = [], []
    for sc in scen:
        s0 = sc["s0"]; ev = bench.evidence(s0, INT_VARIANT); cat = render_catalogue(s0)
        if sc["idx"] < k1:
            for e in sorted(ev):
                d = ev[e]; kind = e.split(":")[0]
                style = rng.choice(a1_eval.STYLES) if kind in MODEL_ITEMS else "structured"
                report = a1_eval.render(s0, e, d, style, rng)
                t = dict(id=f"a1|{sc['sid']}|{e}", agent="a1", sid=sc["sid"], evidence=e, type=d["type"], style=style,
                         model=kind in MODEL_ITEMS, report=report,
                         truth_scope=sorted(set(d["M"]) | set(d["X"])), truth_consumes=d["D"],
                         manifest=d["M"], template=sorted(bench.template(s0, d["type"])),
                         catalogue_keys=sorted(a1_eval.catalogue(s0)), evidence_ids=sorted(ev))
                if t["model"]:
                    t["prompt"] = A1_PROMPT.format(eid=e, report=report, cat=cat,
                                                  evs=", ".join(x for x in sorted(ev) if x != e))
                a1.append(t)
        if sc["idx"] < k3:
            graph = render_graph(s0, ev)
            for ck, s1 in sc["changes"]:
                stale, crit, C = truth_for(s0, s1)
                inv = core_flag(ev, C)
                ecr = describe_change(s0, s1, ck)
                base = dict(sid=sc["sid"], change=ck, ecr=ecr, stale=sorted(stale), crit=sorted(crit),
                            changed=sorted(C), inv=sorted(inv), evidence_ids=sorted(ev),
                            consumes={e: ev[e]["D"] for e in ev},
                            scope={e: sorted(set(ev[e]["M"]) | set(ev[e]["X"])) for e in ev},
                            cost={e: bench.COST_H[ev[e]["type"]] for e in ev})
                a3.append(dict(base, id=f"impact|{sc['sid']}|{ck}", agent="impact",
                               prompt=IMPACT_PROMPT.format(ecr=ecr, cat=cat, graph=graph)))
                a3.append(dict(base, id=f"a3|{sc['sid']}|{ck}", agent="a3",
                               prompt=A3_PROMPT.format(ecr=ecr, cat=cat, graph=graph, changed=", ".join(sorted(C)) or "(none)",
                                                       inv=", ".join(sorted(inv)) or "(empty)")))
    return a1, a3

def request_params(task, model=MODEL, effort=EFFORT):
    """Messages API parameters for one task (used for both batch and direct calls)."""
    return dict(model=model, max_tokens=MAX_TOKENS[task["agent"]], system=SYSTEM,
                thinking={"type": "adaptive"},
                output_config={"effort": effort, "format": {"type": "json_schema", "schema": SCHEMAS[task["agent"]]}},
                messages=[{"role": "user", "content": task["prompt"]}])

def cost_usd(usage, batch=True):
    f = BATCH_DISCOUNT if batch else 1.0
    i = usage.get("input_tokens", 0) + usage.get("cache_creation_input_tokens", 0) * 1.25 + \
        usage.get("cache_read_input_tokens", 0) * 0.1
    return f * (i * PRICE_IN + usage.get("output_tokens", 0) * PRICE_OUT) / 1e6

# ============================================================================ offline proposers
def offline_answer(task, proposer, rng):
    a = task["agent"]
    if a == "a1":
        if proposer == "oracle":
            return {"scope": task["truth_scope"], "consumes": task["truth_consumes"], "rationale": "oracle"}
        if proposer == "parser+templates":
            p = a1_eval.parser(dict(task, catalogue={k: "" for k in task["catalogue_keys"]}))
            p["scope"] = sorted(set(p["scope"]) | set(task["template"]))
            if task["type"] == "rta": p["consumes"] = sorted(set(p["consumes"]) | {"INT"})
            return dict(p, rationale="parser+templates")
        if proposer == "noisy":   # oracle with 20% omissions, for testing the checks
            return {"scope": [x for x in task["truth_scope"] if rng.random() > 0.2],
                    "consumes": task["truth_consumes"], "rationale": "noisy"}
        return {"scope": [], "consumes": [], "rationale": "empty"}
    if a == "impact":
        st = task["inv"] if proposer == "oracle" else []
        return {"stale": [{"evidence": e, "reason": proposer} for e in st]}
    if a == "a3":
        order = topo(task["inv"], task["consumes"], task["cost"])
        items = []
        for e in order:
            first = sorted(set(task["scope"][e]) & set(task["changed"])) or \
                    sorted(set(task["consumes"][e]) & set(task["inv"]))
            items.append({"evidence": e, "path": first[:1] + [e], "rationale": proposer})
        return {"order": order, "items": items, "extra": []}

def topo(items, consumes, cost):
    left, out = set(items), []
    while left:
        ready = sorted([e for e in left if not (set(consumes[e]) & left)], key=lambda e: (cost[e], e))
        if not ready:
            ready = sorted(left)
        out.append(ready[0]); left.discard(ready[0])
    return out

def a1_fill(task):
    """A1 is called only for reports without a machine-readable manifest. Structured unit-test
    and static-analysis reports name their unit literally; the parser handles them."""
    p = a1_eval.parser(dict(task, catalogue={k: "" for k in task["catalogue_keys"]}))
    return dict(p, rationale="parser (structured report)")

# ============================================================================ scoring
def parse_answer(text):
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text or "", re.S)
        try:
            return json.loads(m.group(0)) if m else None
        except Exception:
            return None

def score(a1_tasks, a3_tasks, answers, scen=None):
    """answers: id -> parsed answer dict (None = refusal/error/unparseable)."""
    R = {}
    # ---------------------------------------------------------------- A1 proposal quality
    by = defaultdict(lambda: defaultdict(float))
    props = defaultdict(dict)       # sid -> e -> (scope, consumes, rejected)
    for t in a1_tasks:
        ans = answers.get(t["id"]) if t["model"] else a1_fill(t)
        failed = ans is None
        ans = ans or {"scope": [], "consumes": []}
        sc, co = set(map(str, ans.get("scope", []))), set(map(str, ans.get("consumes", [])))
        true, tc = set(t["truth_scope"]), set(t["truth_consumes"])
        viol = check_proposal(t["manifest"], t["truth_scope"], t["type"], t["template"], sc, co) \
            if not failed else ["no-answer"]
        props[t["sid"]][t["evidence"]] = (sc, co, bool(viol))
        if not t["model"]:
            continue
        omitted, invented = true - sc, sc - true
        unknown = sc - set(t["catalogue_keys"])
        bad = bool(omitted) or bool(tc - co)
        for key in ("ALL", f"type:{t['type']}", f"style:{t['style']}"):
            b = by[key]
            b["n"] += 1; b["true"] += len(true); b["omitted"] += len(omitted); b["invented"] += len(invented)
            b["unknown_keys"] += len(unknown); b["exact"] += (not omitted and not invented and tc == co)
            b["with_omission"] += bad; b["rejected"] += bool(viol); b["bad_rejected"] += bad and bool(viol)
            b["bad_accepted"] += bad and not viol; b["good_rejected"] += (not bad) and bool(viol)
            b["failed"] += failed; b["consume_missed"] += len(tc - co); b["consume_invented"] += len(co - tc)
    for b in by.values():
        n = b["n"]
        b["omission_rate"] = b["omitted"] / b["true"] if b["true"] else None
        b["exact_rate"] = b["exact"] / n
        b["items_with_omission_rate"] = b["with_omission"] / n
        b["rejection_rate"] = b["rejected"] / n
        b["bad_rejected_share"] = b["bad_rejected"] / b["with_omission"] if b["with_omission"] else None
        b["undetected_omission_rate"] = b["bad_accepted"] / n
    R["a1"] = {k: dict(v) for k, v in sorted(by.items())}

    # ---------------------------------------------------------------- A1 end-to-end (stale evidence)
    if scen:
        E = defaultdict(float)
        for sc_ in scen:
            sid = sc_["sid"]
            if sid not in props:
                continue
            s0 = sc_["s0"]; ev = bench.evidence(s0, INT_VARIANT)
            P = props[sid]
            for mode in ("true", "agent_nochecks", "agent_checks"):
                if mode == "true":
                    S = D = None
                else:
                    S, D = {}, {}
                    for e, d in ev.items():
                        sc, co, rej = P[e]
                        if mode == "agent_checks" and rej:
                            # a human corrects a rejected proposal by adding what it omitted; the
                            # agent's additional entries stay (as in bench.perturb_scopes)
                            sc, co = sc | set(d["M"]) | set(d["X"]), co | set(d["D"])
                        S[e], D[e] = sc, co
                for ck, s1 in sc_["changes"]:
                    stale, crit, C = truth_for(s0, s1)
                    f = core_flag(ev, C, S, D)
                    E[f"{mode}.n"] += 1; E[f"{mode}.stale"] += len(stale); E[f"{mode}.miss"] += len(stale - f)
                    E[f"{mode}.critmiss"] += len(crit - f); E[f"{mode}.scen_critmiss"] += bool(crit - f)
                    E[f"{mode}.scen_miss"] += bool(stale - f); E[f"{mode}.fp"] += len(f - stale)
                    E[f"{mode}.hours"] += sum(bench.COST_H[ev[e]["type"]] for e in f)
                    E[f"{mode}.need_h"] += sum(bench.COST_H[ev[e]["type"]] for e in stale)
                    if ck != "dma_enable":
                        E[f"{mode}.nodma_stale"] += len(stale); E[f"{mode}.nodma_miss"] += len(stale - f)
            E["systems"] += 1
            E["escalated"] += sum(1 for e in ev if P[e][2]); E["items"] += len(ev)
        for m in ("true", "agent_nochecks", "agent_checks"):
            if E[f"{m}.stale"]:
                E[f"{m}.recall"] = 1 - E[f"{m}.miss"] / E[f"{m}.stale"]
                E[f"{m}.recall_nodma"] = 1 - E[f"{m}.nodma_miss"] / E[f"{m}.nodma_stale"]
                E[f"{m}.effort_ratio"] = E[f"{m}.hours"] / E[f"{m}.need_h"]
        R["a1_end_to_end"] = dict(E)

    # ---------------------------------------------------------------- IMPACT (model alone) and A3 (with core)
    I = defaultdict(lambda: defaultdict(float))
    A = defaultdict(lambda: defaultdict(float))
    for t in a3_tasks:
        ans = answers.get(t["id"])
        stale, crit, inv = set(t["stale"]), set(t["crit"]), set(t["inv"])
        evs = set(t["evidence_ids"])
        if t["agent"] == "impact":
            for key in ("ALL", f"change:{t['change']}"):
                b = I[key]; b["n"] += 1; b["failed"] += ans is None
                pred = {x.get("evidence") for x in (ans or {}).get("stale", [])} & evs
                b["stale"] += len(stale); b["tp"] += len(pred & stale); b["fp"] += len(pred - stale)
                b["miss"] += len(stale - pred); b["crit"] += len(crit); b["critmiss"] += len(crit - pred)
                b["scen_miss"] += bool(stale - pred); b["scen_critmiss"] += bool(crit - pred)
                b["unknown_ids"] += len({x.get("evidence") for x in (ans or {}).get("stale", [])} - evs)
                b["core_miss"] += len(stale - inv); b["core_critmiss"] += len(crit - inv)
                b["hours"] += sum(t["cost"][e] for e in pred); b["need_h"] += sum(t["cost"][e] for e in stale)
                b["core_hours"] += sum(t["cost"][e] for e in inv)
        else:
            ans = ans or {}
            order = [str(x) for x in ans.get("order", [])]
            pos = {e: i for i, e in enumerate(order)}
            complete = set(order) == inv and len(order) == len(inv)
            dep_ok = all(pos[d] < pos[e] for e in order if e in inv for d in t["consumes"].get(e, []) if d in inv and d in pos)
            valid = complete and dep_ok and bool(ans)
            items = {x.get("evidence"): x for x in ans.get("items", [])}
            path_ok = path_n = 0
            for e in inv:
                x = items.get(e); path_n += 1
                if not x or not x.get("path"):
                    continue
                p0 = x["path"][0]
                if (p0 in t["changed"] and p0 in t["scope"][e]) or (p0 in inv and p0 in t["consumes"][e]):
                    path_ok += 1
            extra = {x.get("evidence") for x in ans.get("extra", [])} & evs
            missed_by_core = stale - inv
            for key in ("ALL", f"change:{t['change']}"):
                b = A[key]; b["n"] += 1; b["failed"] += not ans
                b["plan_valid"] += valid; b["plan_incomplete"] += (not complete) and bool(ans)
                b["plan_dep_violation"] += (not dep_ok) and bool(ans)
                b["path_ok"] += path_ok; b["path_n"] += path_n
                b["extra"] += len(extra); b["extra_stale"] += len(extra & missed_by_core)
                b["extra_redundant"] += len(extra & inv)
                b["core_miss"] += len(missed_by_core); b["core_critmiss"] += len(crit - inv)
                b["union_miss"] += len(missed_by_core - extra); b["union_critmiss"] += len(crit - inv - extra)
                b["extra_hours"] += sum(t["cost"][e] for e in extra - inv)
                b["core_hours"] += sum(t["cost"][e] for e in inv); b["stale"] += len(stale)
                b["need_h"] += sum(t["cost"][e] for e in stale)
    for b in I.values():
        b["recall"] = b["tp"] / b["stale"] if b["stale"] else None
        b["precision"] = b["tp"] / (b["tp"] + b["fp"]) if (b["tp"] + b["fp"]) else None
        b["core_recall"] = 1 - b["core_miss"] / b["stale"] if b["stale"] else None
        b["effort_ratio"] = b["hours"] / b["need_h"] if b["need_h"] else None
    for b in A.values():
        b["plan_valid_rate"] = b["plan_valid"] / b["n"]
        b["path_ok_rate"] = b["path_ok"] / b["path_n"] if b["path_n"] else None
        b["core_recall"] = 1 - b["core_miss"] / b["stale"] if b["stale"] else None
        b["union_recall"] = 1 - b["union_miss"] / b["stale"] if b["stale"] else None
    R["impact"] = {k: dict(v) for k, v in sorted(I.items())}
    R["a3"] = {k: dict(v) for k, v in sorted(A.items())}
    return R

# ============================================================================ CLI
def _write(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

def _read(path):
    return [json.loads(l) for l in open(path)]

if __name__ == "__main__":
    cmd, out = sys.argv[1], sys.argv[2]
    os.makedirs(out, exist_ok=True)
    if cmd == "tasks":
        a1, a3 = gen_tasks(int(sys.argv[3]), int(sys.argv[4]))
        _write(f"{out}/a1_tasks.jsonl", a1); _write(f"{out}/a3_tasks.jsonl", a3)
        n = sum(t["model"] for t in a1) + len(a3)
        chars = sum(len(t.get("prompt", "")) for t in a1 + a3)
        print(json.dumps({"a1_tasks": len(a1), "a1_model_calls": sum(t["model"] for t in a1),
                          "a3_impact_calls": len(a3), "model_calls": n, "prompt_chars": chars,
                          "approx_input_tokens": chars // 4}))
    elif cmd == "offline":
        prop = sys.argv[3]; rng = random.Random(0)
        rows = [{"id": t["id"], "answer": offline_answer(t, prop, rng)}
                for t in _read(f"{out}/a1_tasks.jsonl") + _read(f"{out}/a3_tasks.jsonl") if t.get("prompt")]
        _write(f"{out}/answers_{prop}.jsonl", rows)
    elif cmd == "score":
        a1, a3 = _read(f"{out}/a1_tasks.jsonl"), _read(f"{out}/a3_tasks.jsonl")
        ans = {r["id"]: r["answer"] for r in _read(sys.argv[3])}
        k = max(int(t["sid"].split("sys")[1]) for t in a1 + a3) + 1
        json.dump(score(a1, a3, ans, sample_scenarios(k)), sys.stdout, indent=1, default=float)

In [ ]:
import anthropic, agents
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
agents.MODEL = MODEL
client = anthropic.Anthropic()
m = client.models.retrieve(MODEL)          # free; fails early if the key or model id is wrong
print('Model OK:', m.id, '| anthropic', anthropic.__version__)

K1MAX, K3MAX = 10, 4                       # largest sample the planner may choose (systems per seed)
A1, A3 = agents.gen_tasks(K1MAX, K3MAX)
TASKS = {t['id']: t for t in A1 + A3 if t.get('prompt')}
idx = lambda t: int(t['sid'].split('sys')[1])
print(len(TASKS), 'model tasks available')

STATE = f'{OUT}/state.json'; ANS = f'{OUT}/answers.jsonl'
state = json.load(open(STATE)) if os.path.exists(STATE) else {'batches': {}}
def save_state(): json.dump(state, open(STATE, 'w'), indent=1)
def answers():
    out = {}
    if os.path.exists(ANS):
        for l in open(ANS):
            r = json.loads(l); out[r['id']] = r
    return out
def spent(): return sum(r['cost_usd'] for r in answers().values())

In [ ]:
def run_batch(name, ids):
    """Submit (or resume) one batch, wait for it, append results to answers.jsonl."""
    done = answers()
    ids = [i for i in ids if i not in done or done[i]['status'] == 'errored']
    if not ids:
        print(name, 'already done'); return
    if name not in state['batches']:
        reqs = [Request(custom_id=f'r{n}', params=MessageCreateParamsNonStreaming(**agents.request_params(TASKS[i], MODEL)))
                for n, i in enumerate(ids)]
        b = client.messages.batches.create(requests=reqs)
        state['batches'][name] = {'id': b.id, 'map': {f'r{n}': i for n, i in enumerate(ids)}}
        save_state(); print(name, 'submitted', b.id, len(ids), 'requests')
    info = state['batches'][name]
    while True:
        b = client.messages.batches.retrieve(info['id'])
        if b.processing_status == 'ended': break
        print(time.strftime('%H:%M:%S'), name, b.request_counts); time.sleep(60)
    with open(ANS, 'a') as f:
        for r in client.messages.batches.results(info['id']):
            tid = info['map'][r.custom_id]
            row = {'id': tid, 'status': r.result.type, 'answer': None, 'text': None, 'stop_reason': None,
                   'usage': {}, 'cost_usd': 0.0, 'model': None}
            if r.result.type == 'succeeded':
                msg = r.result.message
                text = ''.join(c.text for c in msg.content if c.type == 'text')
                u = msg.usage.model_dump(exclude_none=True)
                u = {k: v for k, v in u.items() if isinstance(v, (int, float))}
                row.update(text=text, stop_reason=msg.stop_reason, usage=u, model=msg.model,
                           cost_usd=agents.cost_usd(u, batch=True),
                           answer=agents.parse_answer(text) if msg.stop_reason == 'end_turn' else None)
            f.write(json.dumps(row) + '\n')
    print(name, 'finished; total spent so far $%.2f' % spent())

## 1. Pilot (one system) to measure the real cost per call

In [ ]:
pilot = [i for i, t in TASKS.items() if t['sid'] == 'seed1.sys0']
run_batch('pilot', pilot)
A = answers()
per = {}
for ag in ('a1', 'impact', 'a3'):
    c = [A[i]['cost_usd'] for i in pilot if TASKS[i]['agent'] == ag and i in A]
    per[ag] = sum(c) / max(1, len(c))
print('mean cost per call (USD):', {k: round(v, 4) for k, v in per.items()})
fails = [i for i in pilot if A.get(i, {}).get('answer') is None]
print('pilot calls without a usable answer:', len(fails), fails[:5])

## 2. Choose the sample size that fits the budget, then run it

In [ ]:
def plan_cost(k1, k3):
    ids = [i for i, t in TASKS.items() if (t['agent'] == 'a1' and idx(t) < k1) or (t['agent'] != 'a1' and idx(t) < k3)]
    todo = [i for i in ids if i not in answers()]
    return ids, sum(per[TASKS[i]['agent']] for i in todo) * 1.25   # 25% safety margin
plans = [(4, 1), (6, 1), (6, 2), (8, 2), (8, 3), (10, 3), (10, 4)]
left = BUDGET_USD - spent()
chosen = None
for k1, k3 in plans:
    ids, c = plan_cost(k1, k3)
    print(f'K1={k1} K3={k3}: {len(ids)} calls, est. ${c:.2f} more (left ${left:.2f})')
    if c <= left: chosen = (k1, k3, ids)
assert chosen, 'Budget too small even for the smallest plan; raise BUDGET_USD.'
if state.get('plan'):                      # a resumed run keeps the plan it started with
    K1, K3 = state['plan']; ids = plan_cost(K1, K3)[0]
else:
    K1, K3, ids = chosen
state['plan'] = [K1, K3]; save_state()
print('Running K1 =', K1, 'and K3 =', K3, 'systems per seed (x3 seeds)')
run_batch(f'main_{K1}_{K3}', ids)
run_batch(f'retry_{K1}_{K3}', [i for i in ids if answers().get(i, {}).get('status') == 'errored'])
print('TOTAL SPENT: $%.2f' % spent())

## 3. Score and save

In [ ]:
A = answers()
ans = {i: r['answer'] for i, r in A.items()}
a1 = [t for t in A1 if idx(t) < K1]; a3 = [t for t in A3 if idx(t) < K3]
scen = agents.sample_scenarios(K1)
S = agents.score(a1, a3, ans, scen)
S['meta'] = {'model': MODEL, 'served_models': sorted({r['model'] for r in A.values() if r['model']}),
             'effort': agents.EFFORT, 'K1': K1, 'K3': K3, 'calls': len(A), 'spent_usd': round(spent(), 2),
             'stop_reasons': {s: sum(1 for r in A.values() if r['stop_reason'] == s) for s in {r['stop_reason'] for r in A.values()}},
             'anthropic_sdk': anthropic.__version__, 'date': time.strftime('%Y-%m-%d')}
json.dump(S, open(f'{OUT}/scores.json', 'w'), indent=1, default=float)
a, e, i3, p3 = S['a1']['ALL'], S['a1_end_to_end'], S['impact']['ALL'], S['a3']['ALL']
print(f"A1: {int(a['n'])} proposals, omission rate {a['omission_rate']:.3f}, exact {a['exact_rate']:.2f}, "
      f"rejected by checks {a['rejection_rate']:.2f}, bad proposals rejected {a['bad_rejected_share']}")
print(f"A1 end-to-end recall: true scopes {e['true.recall']:.3f}, agent no checks {e['agent_nochecks.recall']:.3f}, agent + checks {e['agent_checks.recall']:.3f}")
print(f"Model alone (impact): recall {i3['recall']:.3f}, precision {i3['precision']:.3f}, AAG core recall {i3['core_recall']:.3f}")
print(f"A3 plans valid {p3['plan_valid_rate']:.2f}, causal paths correct {p3['path_ok_rate']:.2f}, extras that caught core misses {int(p3['extra_stale'])}")
print('Saved to', OUT, '- tell Claude it has finished.')